# 로컬 AI 9강 — 한 문장이 한 장의 그림으로

「작은 로봇이 유리 빌딩 도시 위로 노을을 그린다」 — 이 한 문장이 어떻게 그림 한 장이 될까요?
오늘은 가볍고 빠른 **SD-Turbo** 로 그림 모델의 속을 한 칸씩 열어 봅니다. 그리고 더 큰 최신 모델 **Qwen-Image 2.1 Turbo** 를 내 컴퓨터에서 돌리는 길까지 갑니다.

| 셀 | 하는 일 |
|---|---|
| 1 | 준비 — 내 코랩 자원 확인 |
| 2 | 모델 받기 — 세 부품(글 이해 · 그림 그리기 · 펼치기) |
| 3 | 📝 문장을 숫자로 — 토큰과 77칸 |
| 4 | 🌫️ 노이즈에서 시작 — 48배 작은 잠재 공간 |
| 5 | 👣 네 걸음 지켜보기 — 노이즈가 그림이 되는 순간 |
| 6 | 🗜️ VAE — 압축했다가 다시 펼치기 |
| 7 | 🖼️ 첫 그림 한 장 — 한 걸음 |
| 8 | 🎲 seed 는 주사위 |
| 9 | ✏️ 단어 하나 바꾸기 |
| 10 | ⏱️ 걸음 수 실험 — 많이 걸을수록 좋을까? |
| 11 | 💾 파일 크기 = 파라미터 × 바이트 — 내 컴퓨터에 들어갈까 |
| 12 | 🏠 큰 모델은 내 컴퓨터로 — Qwen-Image 2.1 Turbo |
| 13 | 📄 모델 카드와 라이선스 읽기 |
| 14 | 🧾 기록 남기기 |

> 런타임 → 런타임 유형 변경 → **T4 GPU** 를 고른 뒤 위에서부터 순서대로 ▶ 실행하세요. (런타임 → 모두 실행 도 됩니다. 처음 모델 받기에 2~4분)
> GPU 가 없어도 돌아갑니다. CPU 로는 한 장에 수십 초가 걸려요.

## 1. 준비 — 내 코랩 자원 확인

그림 모델은 **GPU 메모리(VRAM)**, **시스템 메모리(RAM)**, **디스크** 세 곳을 씁니다. 셋은 서로 다른 공간이에요.

In [ ]:
!pip install -q diffusers==0.41.0 transformers==5.19.0 accelerate safetensors

import os, time, json, shutil
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display

if torch.cuda.is_available():
    장치 = "cuda"; 이름 = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
elif torch.backends.mps.is_available():
    장치 = "mps"; 이름 = "Apple Silicon (MPS)"; vram = None
else:
    장치 = "cpu"; 이름 = "CPU 만"; vram = None
ram = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1024**3
disk = shutil.disk_usage("/").free / 1024**3
정밀도 = torch.float16 if 장치 != "cpu" else torch.float32
plt.rcParams["figure.dpi"] = 100

print(f"계산 장치 : {장치} · {이름}")
print(f"GPU 메모리: {vram:.1f} GiB" if vram else "GPU 메모리: (없음 또는 RAM 과 공유)")
print(f"시스템 RAM: {ram:.1f} GiB · 남은 디스크 {disk:.0f} GiB")
print("숫자 정밀도:", "16비트(fp16)" if 정밀도 == torch.float16 else "32비트(fp32)")

## 2. 모델 받기 — 세 부품

그림 모델 하나는 사실 **세 부품**입니다.

| 부품 | 하는 일 | 비유 |
|---|---|---|
| 글 이해 (text encoder) | 문장 → 숫자 | 주문을 받아 적는 사람 |
| 그림 그리기 (U-Net) | 노이즈에서 그림을 조금씩 꺼냄 | 화가 |
| 펼치기 (VAE) | 작은 잠재 그림 → 큰 픽셀 그림 | 인화기 |

SD-Turbo 는 Stability AI 가 만든 **한 걸음(1 step)** 용 모델입니다. 512×512 그림을 만듭니다.

> 실행하면 「safety checker 가 꺼져 있다」는 영어 경고가 뜹니다. SD-Turbo 저장소에는 결과를 걸러 주는 안전 검사 부품이 들어 있지 않아서예요.
> 공부용으로 내 컴퓨터에서 쓰는 데는 문제가 없지만, **다른 사람이 쓰는 서비스에 붙일 때는 결과를 거르는 장치를 따로** 둬야 합니다.

In [ ]:
from diffusers import AutoPipelineForText2Image

모델 = "stabilityai/sd-turbo"
판 = "b261bac6fd2cf515557d5d0707481eafa0485ec2"   # 같은 결과를 위해 버전을 고정
시작 = time.time()
옵션 = dict(revision=판, torch_dtype=정밀도)
if 정밀도 == torch.float16:
    옵션["variant"] = "fp16"            # 16비트 파일만 받기 (절반 크기)
pipe = AutoPipelineForText2Image.from_pretrained(모델, **옵션).to(장치)
pipe.set_progress_bar_config(disable=True)
print(f"받고 올리기 {time.time()-시작:.0f}초")

부품 = {"글 이해 (text encoder)": pipe.text_encoder, "그림 그리기 (U-Net)": pipe.unet, "펼치기 (VAE)": pipe.vae}
합 = 0
for k, m in 부품.items():
    n = sum(p.numel() for p in m.parameters()); 합 += n
    print(f"{k:24s} {n/1e6:8.1f} 백만 개 숫자")
print(f"{'합계':24s} {합/1e9:8.2f} 십억 개 (1.29B)")

## 3. 📝 문장을 숫자로 — 토큰과 77칸

그림 모델은 글자를 못 읽습니다. 먼저 문장을 **토큰**(낱말 조각)으로 자르고, 토큰마다 번호를 붙입니다.
그다음 글 이해 부품이 **77칸 × 1024개 숫자**로 문장의 뜻을 만듭니다. 77칸보다 긴 문장은 뒤가 잘려요.

In [ ]:
문장 = "a small robot painting a sunset over a city of glass towers, warm light, detailed"

토큰 = pipe.tokenizer(문장)["input_ids"]
조각 = pipe.tokenizer.convert_ids_to_tokens(토큰)
print(f"토큰 {len(토큰)}개 (77칸 중)")
for t, i in zip(조각, 토큰):
    print(f"  {i:6d}  {t}")

with torch.no_grad():
    뜻 = pipe.encode_prompt(문장, 장치, 1, False)[0]
print("\n문장의 뜻(임베딩) 모양:", tuple(뜻.shape), "→ 77칸 × 1024 숫자")

plt.figure(figsize=(10, 3.2))
plt.imshow(뜻[0, :len(토큰), :96].float().cpu().numpy(), aspect="auto", cmap="magma")
plt.yticks(range(len(조각)), [t.replace("</w>", "") for t in 조각], fontsize=7)
plt.xlabel("1024 numbers per token (first 96 shown)"); plt.title("The sentence as numbers")
plt.tight_layout(); plt.show()

## 4. 🌫️ 노이즈에서 시작 — 48배 작은 잠재 공간

그림은 **완전한 노이즈**에서 시작합니다. 그런데 512×512 픽셀(786,432개 숫자)에 바로 그리지 않고,
**4×64×64 = 16,384개 숫자**짜리 작은 「잠재 공간」에서 그립니다. **48배** 작아서 빠르고 가볍습니다.
노이즈를 만드는 주사위 번호가 **seed** 입니다.

In [ ]:
SEED = 8
g = torch.Generator("cpu").manual_seed(SEED)
노이즈 = torch.randn((1, 4, 64, 64), generator=g)
print("잠재 노이즈 모양:", tuple(노이즈.shape), "=", 노이즈.numel(), "개 숫자")
print("최종 그림 픽셀  :", 512*512*3, "개 숫자 →", 512*512*3 // 노이즈.numel(), "배 작다")

fig, ax = plt.subplots(1, 4, figsize=(10, 2.8))
for c in range(4):
    ax[c].imshow(노이즈[0, c].numpy(), cmap="gray"); ax[c].set_title(f"channel {c+1} · 64×64"); ax[c].axis("off")
plt.suptitle(f"Starting noise (seed {SEED})"); plt.tight_layout(); plt.show()

## 5. 👣 네 걸음 지켜보기 — 노이즈가 그림이 되는 순간

화가(U-Net)는 매 걸음마다 「여기 섞인 노이즈는 이만큼」이라고 **예측해서 뺍니다**. 문장의 뜻(3번 셀)을 보면서요.
걸음마다 잠재 그림을 펼쳐(VAE) 화면에 찍어 봅니다.

In [ ]:
def 펼치기(잠재):
    with torch.no_grad():
        x = pipe.vae.decode(잠재.to(pipe.vae.dtype) / pipe.vae.config.scaling_factor).sample
    return ((x[0].float().clamp(-1, 1) + 1) / 2).permute(1, 2, 0).cpu().numpy()

장면 = []
def 걸음마다(p, i, t, kw):
    장면.append(펼치기(kw["latents"])); return kw

g = torch.Generator("cpu").manual_seed(SEED)
pipe(문장, num_inference_steps=4, guidance_scale=0.0, generator=g,
     callback_on_step_end=걸음마다, callback_on_step_end_tensor_inputs=["latents"])
처음 = 펼치기(노이즈.to(장치) * pipe.scheduler.init_noise_sigma)

fig, ax = plt.subplots(1, 5, figsize=(14, 3.2))
for a, img, t in zip(ax, [처음] + 장면, ["noise", "step 1/4", "step 2/4", "step 3/4", "step 4/4"]):
    a.imshow(img); a.set_title(t); a.axis("off")
plt.tight_layout(); plt.show()

## 6. 🗜️ VAE — 압축했다가 다시 펼치기

VAE 는 그림을 **48배 작게 압축**(encode)했다가 다시 **펼치는**(decode) 부품입니다.
방금 만든 그림을 압축했다가 펼쳐서, 얼마나 그대로 돌아오는지 봅니다.

In [ ]:
원본 = (장면[-1] * 255).astype("uint8")
x = torch.from_numpy(원본).permute(2, 0, 1)[None].float().div(127.5).sub(1).to(장치, pipe.vae.dtype)
with torch.no_grad():
    잠재 = pipe.vae.encode(x).latent_dist.mean
복원 = 펼치기(잠재 * pipe.vae.config.scaling_factor)
차이 = np.abs(복원 - 원본 / 255).mean()
print("픽셀:", x.numel(), "개 → 잠재:", 잠재.numel(), "개 → 다시 픽셀")
print(f"평균 차이 {차이*100:.1f}% (0% 면 완벽하게 같음)")

fig, ax = plt.subplots(1, 3, figsize=(10, 3.4))
ax[0].imshow(원본); ax[0].set_title("original 512×512")
잠재그림 = 잠재[0, :3].float().cpu().permute(1, 2, 0).numpy()
잠재그림 = (잠재그림 - 잠재그림.min()) / (잠재그림.max() - 잠재그림.min())
ax[1].imshow(잠재그림); ax[1].set_title("latent 64×64 (3 of 4 ch)")
ax[2].imshow(복원); ax[2].set_title("decoded again")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

## 7. 🖼️ 첫 그림 한 장 — 한 걸음

SD-Turbo 는 **한 걸음**에 그리도록 훈련된 모델입니다. 문장을 바꿔서 여러 번 실행해 보세요.

In [ ]:
def 그리기(문장, seed=8, 걸음=1):
    g = torch.Generator("cpu").manual_seed(seed)
    t = time.time()
    img = pipe(문장, num_inference_steps=걸음, guidance_scale=0.0, generator=g).images[0]
    return img, time.time() - t

그리기(문장)                      # 한 번은 데우기 (첫 실행은 느림)
그림, 초 = 그리기(문장)
print(f"{초:.2f}초 · seed 8 · 1걸음")
display(그림)

## 8. 🎲 seed 는 주사위

같은 문장이라도 seed(시작 노이즈)가 다르면 다른 그림이 나옵니다.
반대로 **seed 가 같으면 같은 그림**이 나와요. 마음에 든 그림은 seed 를 꼭 적어 두세요.

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(13, 3.6))
for a, s in zip(ax, [1, 2, 3, 4]):
    a.imshow(그리기(문장, seed=s)[0]); a.set_title(f"seed {s}"); a.axis("off")
plt.tight_layout(); plt.show()

a = np.array(그리기(문장, seed=8)[0]); b = np.array(그리기(문장, seed=8)[0])
print(f"seed 8 로 두 번 → 픽셀 일치 {(a == b).mean()*100:.1f}%")
print("※ 다른 GPU·다른 버전에서는 같은 seed 라도 조금 달라질 수 있어요.")

## 9. ✏️ 단어 하나 바꾸기

seed 는 그대로 두고 **단어 하나**만 바꿉니다. 구도는 비슷하게 남고, 바꾼 단어의 뜻만 바뀌는 걸 보세요.

In [ ]:
낱말 = ["sunset", "snowstorm", "neon night"]
fig, ax = plt.subplots(1, 3, figsize=(11, 4))
for a, w in zip(ax, 낱말):
    a.imshow(그리기(문장.replace("sunset", w), seed=8)[0]); a.set_title(w); a.axis("off")
plt.tight_layout(); plt.show()

## 10. ⏱️ 걸음 수 실험 — 많이 걸을수록 좋을까?

보통 그림 모델은 20~50걸음을 걷습니다. Turbo 모델은 **한 걸음에 맞춰 훈련(증류)** 된 모델이에요.
걸음을 늘리면 시간은 늘고, 그림은 오히려 달라지거나 뭉개질 수 있습니다. 직접 보세요.

In [ ]:
걸음들 = [1, 2, 4]
fig, ax = plt.subplots(1, 3, figsize=(11, 4))
for a, n in zip(ax, 걸음들):
    img, 초 = 그리기(문장, seed=8, 걸음=n)
    a.imshow(img); a.set_title(f"{n} step(s) · {초:.2f}s"); a.axis("off")
plt.tight_layout(); plt.show()
print("모델마다 권장 걸음 수가 다릅니다 — SD-Turbo 1걸음, Qwen-Image 2.1 Turbo 8걸음. 모델 카드에 적혀 있어요.")

## 11. 💾 파일 크기 = 파라미터 × 바이트 — 내 컴퓨터에 들어갈까

모델 파일 크기는 거의 **숫자 개수 × 숫자 하나의 크기**입니다.
32비트 = 4바이트, 16비트 = 2바이트, 8비트 = 1바이트, 4비트 = 0.5바이트. 비트를 줄이는 것을 **양자화**라고 합니다.

In [ ]:
def 크기_GB(파라미터_십억, 비트):
    return 파라미터_십억 * 1e9 * 비트 / 8 / 1e9

모델들 = {"SD-Turbo (1.29B)": 1.29, "Qwen-Image 2.1 그림 (7B)": 7.0, "Qwen-Image 2.1 글 이해 (8B)": 8.0}
print(f"{'':28s}{'32비트':>8s}{'16비트':>8s}{'8비트':>8s}{'4비트':>8s}  (GB)")
for k, b in 모델들.items():
    print(f"{k:28s}" + "".join(f"{크기_GB(b, n):8.1f}" for n in [32, 16, 8, 4]))

print("\n실제 파일 (Qwen-Image 2.1 Turbo, 4비트 묶음): 그림 4.20 + 글 이해 6.31 + 펼치기 0.68 = 약 11.2 GB")
print("원본 16비트 묶음은 약 32.5 GB — 무료 코랩 T4(15 GB)에는 원본이 들어가지 않습니다.")
여유 = vram if vram else ram
print(f"\n지금 이 기계: {'GPU' if vram else 'RAM'} {여유:.1f} GiB →",
      "4비트 Qwen 묶음(11.2 GB)이 들어갈 만합니다" if 여유 >= 12 else "4비트 Qwen 도 빠듯합니다 — SD-Turbo 로 연습하세요")

## 12. 🏠 큰 모델은 내 컴퓨터로 — Qwen-Image 2.1 Turbo

Qwen-Image 2.1 Turbo 는 글자까지 또렷하게 그리는 최신 모델입니다 (8걸음 · CFG 1).
원본이 커서 코랩 무료 GPU 로는 어렵고, **4비트 파일로 내 컴퓨터에서** 돌리는 게 현실적입니다.

**엔비디아 GPU 윈도우 PC → ComfyUI**
1. ComfyUI Desktop 설치 → 확장 관리자에서 **ComfyUI-GGUF** 설치
2. 아래 세 파일을 받아 폴더에 넣기
   - `models/unet/` ← `qwen-image-2.1-turbo-Q4_K_M.gguf` (4.2 GB)
   - `models/text_encoders/` ← `qwen3vl_8b_w4a8.safetensors` (6.3 GB) — Comfy 공식 파일을 쓰세요. 글자만 든 다른 GGUF 는 오류가 납니다.
   - `models/vae/` ← `qwen_image_2.1_vae_bf16.safetensors` (0.7 GB)
3. 교재 페이지의 **워크플로 파일**을 ComfyUI 화면에 끌어다 놓고 ▶ 실행

**맥 (Apple Silicon) → mflux 한 줄** (16 GB 맥에서 1024px 한 장 약 2분 30초)
```
pip install mflux
mflux-generate-qwen-2.1 --model mlx-community/Qwen-Image-2.1-mflux-q4 --base-model qwen-image-2.1 --low-ram --prompt "..."
```
맥에서 ComfyUI 를 쓸 때는 **`--cpu-vae`** 옵션을 붙이세요. 맥 GPU 로 이 VAE 를 돌리면 회색 그림만 나옵니다.

In [ ]:
# 받을 파일 주소 (내 컴퓨터에서 받으세요 — 여기서는 주소만 보여 줍니다)
파일 = {
    "models/unet": "https://huggingface.co/unsloth/Qwen-Image-2.1-Turbo-GGUF",
    "models/text_encoders": "https://huggingface.co/Comfy-Org/Qwen-Image-2.1",
    "models/vae": "https://huggingface.co/Comfy-Org/Qwen-Image-2.1 (vae/ 폴더)",
}
for 폴더, 주소 in 파일.items():
    print(f"{폴더:22s} ← {주소}")

## 13. 📄 모델 카드와 라이선스 읽기

모델을 고르기 전에 **모델 카드**(설명서)와 **라이선스**를 먼저 봅니다. 같은 「무료 다운로드」라도 쓸 수 있는 범위가 다릅니다.

| 확인할 것 | SD-Turbo | Qwen-Image 2.1 Turbo |
|---|---|---|
| 라이선스 | Stability AI 라이선스 (매출 규모에 따라 조건) | Qwen Research License (연구·비상업 중심) |
| 권장 해상도·걸음 | 512px · 1~4걸음 | 1024px 안팎 · 8걸음 · CFG 1 |
| 실행 조건 | 코랩 T4, CPU 도 가능 | 4비트 파일로 16 GB 이상 |

허깅페이스에서 이름에 **Uncensored** 가 붙은 모델도 보게 됩니다. 원래 모델의 안전 조정을 바꿨다는 **표기**일 뿐, 무엇을 어떻게 바꿨는지는 모델마다 다릅니다.
이름만 보고 고르지 말고 **누가 만들었는지 · 원본 라이선스 · 바꾼 내용**을 모델 카드에서 확인하세요. 사람·저작권·법이 걸린 그림은 모델과 상관없이 만드는 사람의 책임입니다.

In [ ]:
from huggingface_hub import model_info
for 모델이름 in ["stabilityai/sd-turbo", "Qwen/Qwen-Image-2.1-Turbo"]:
    try:
        info = model_info(모델이름)
        카드 = info.card_data or {}
        표기 = " ".join(str(x) for x in [카드.get("license"), 카드.get("license_name")] if x)
        print(모델이름)
        print("  라이선스 표기:", 표기 or "카드 머리에 없음 → 모델 카드 본문과 LICENSE 파일을 직접 읽으세요")
        print("  태그:", ", ".join((info.tags or [])[:8]))
        print("  카드 주소: https://huggingface.co/" + 모델이름, "\n")
    except Exception as e:
        print(모델이름, "→ 카드를 못 읽었습니다:", e)

## 14. 🧾 기록 남기기

그림만 남기지 말고 **모델 · 버전 · 문장 · seed · 걸음 · 시간 · 장치**를 같이 남기세요. 그래야 다시 만들 수 있습니다.

In [ ]:
os.makedirs("9강_결과", exist_ok=True)
그림, 초 = 그리기(문장, seed=8)
그림.save("9강_결과/seed8.png")
기록 = {"모델": 모델, "버전": 판, "문장": 문장, "seed": 8, "걸음": 1, "초": round(초, 2),
        "장치": 이름, "정밀도": str(정밀도)}
json.dump(기록, open("9강_결과/기록.json", "w"), ensure_ascii=False, indent=2)
print(json.dumps(기록, ensure_ascii=False, indent=2))
shutil.make_archive("9강_결과", "zip", "9강_결과")
try:
    from google.colab import files; files.download("9강_결과.zip")
except ImportError:
    print("9강_결과.zip 저장됨")